# Eye + head gate: train and test, 5-fold CV

The gaze-only gate collapsed its two outputs onto one curve (r = 0.97 between predicted
`S_frame` and `S_gaze`) and never predicted REFIXATION. This notebook trains the next idea:
add **head motion from the IMU** and the **direction agreement between eye and head**, and
classify the four classes directly.

```
eye_speed     (9) ─► causal CNN ─► f_eye(k)  ┐
head_speed    (9) ─► causal CNN ─► f_head(k) ├─ concat per step ─► MLP ─► masked mean+max ─► 4 classes
cos(eye,head) (9) ─► causal CNN ─► f_cos(k)  ┘
```

| signal | from | meaning |
|---|---|---|
| eye_speed | `gaze_rates_window` | √(eye yaw rate² + eye pitch rate²), rad/s |
| head_speed | `imu_window` gyro, in the eye's yaw/pitch axes | √(head yaw rate² + head pitch rate²), rad/s |
| cos(eye, head) | both | +1 same direction (gaze shift), −1 eyes counter-rotating (VOR), 0 when either is still |

Loss: class-weighted cross-entropy. No frames and no DINOv2 features are used.

## What the physiology predicts

| class | eye | head | cos |
|---|---|---|---|
| STABLE | slow | still | 0 |
| REFIXATION | saccade spike | still | 0 |
| PURSUIT | moving | moving | ≈ −1 |
| TRANSITION | saccade spike | moving | +1, then −1 |

Section 4 checks this on the data **before** any training.

## Run order

1. Clone, mount
2. Check the folds, thresholds and IMU coverage
3. IMU axes: factory calibration if a VRS is available, otherwise fitted from data
4. Feature check: does each class look the way the table says?
5. **Train** the full model, 5 folds
6. **Ablations**: eye only, eye + head without cos, and the shuffle control
7. **Test**, once, against the previous gate on the same rows

Everything is written to `MyDrive/GazeVLM/runs/`, so it survives the runtime. The model has
about 5k parameters; a CPU runtime is enough, a GPU just makes it faster.

## 1. Clone, mount, set paths

The code lives on the `eyehead-gate` branch until it is merged; change `BRANCH` to `main`
after that. If `/content/GazeVLM` already exists from another notebook, the cell switches it
to this branch.

In [ ]:
import os, sys, json, time, subprocess

REPO   = '/content/GazeVLM'
URL    = 'https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git'
BRANCH = 'eyehead-gate'          # set to 'main' once the branch is merged

if os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
assert os.path.exists('eyehead/train.py'), f'eyehead/ is not on branch {BRANCH!r}'

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/GazeVLM'
FOLDS     = os.path.join(DRIVE_DIR, 'folds')
THRESH    = os.path.join(DRIVE_DIR, 'thresholds.json')
RUNS      = os.path.join(DRIVE_DIR, 'runs')
RUN_MAIN  = os.path.join(RUNS, 'eyehead')
BASELINE  = os.path.join(DRIVE_DIR, 'release', 'gate_v1', 'test_preds.csv')
IMU_CALIB = os.path.join(DRIVE_DIR, 'imu_calib.json')
os.makedirs(RUNS, exist_ok=True)


def sh(cmd, tail=None):
    """Run a shell command and stream its output into the cell (or only the last lines)."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    keep = []
    for line in p.stdout:
        if tail is None:
            print(line, end='')
        else:
            keep = (keep + [line])[-tail:]
    if tail is not None:
        print(''.join(keep), end='')
    if p.wait():
        raise RuntimeError(f'command failed ({p.returncode}): {cmd}')


print('torch', torch.__version__, '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'CPU (fine: the model is ~5k parameters)')

## 2. Check the folds, the thresholds and IMU coverage

Rows with no usable IMU are dropped from training and validation by default
(`--missing_imu drop`), for every model including the eye-only ablation, so all variants
see the same rows. At test time every row is kept (head = 0 where the IMU is missing), so
the comparison with the previous gate is on identical rows.

In [ ]:
need = [f'train_val_fold{k}.csv' for k in range(1, 6)] + ['test.csv']
missing = [f for f in need if not os.path.exists(os.path.join(FOLDS, f))]
assert not missing, f'missing in {FOLDS}: {missing}\nRun colab_label_quadrants.ipynb first.'
assert os.path.exists(THRESH), f'missing {THRESH}'
th = json.load(open(THRESH))
print(f"tau_frame {th['tau_frame']:.4f}   tau_gaze {th['tau_gaze']:.4f}\n")

from eyehead.features import load_rows, raw_signals, NAMES

cov = []
for f in ('train_val_fold1.csv', 'test.csv'):
    d = load_rows(os.path.join(FOLDS, f))
    E, G, M, ok = raw_signals(d)
    per_vid = pd.Series(ok).groupby(d.sequence.to_numpy()).mean()
    cov.append(dict(file=f, rows=len(d), videos=d.sequence.nunique(),
                    gaze_present=f'{100*M.any(1).mean():.1f}%',
                    imu_present=f'{100*ok.mean():.1f}%',
                    videos_without_imu=int((per_vid == 0).sum())))
    if f == 'test.csv':
        print('test class balance:')
        display(d.quad.map(NAMES).value_counts().rename('rows').to_frame()
                .assign(pct=lambda x: (100 * x.rows / len(d)).round(1)))
display(pd.DataFrame(cov).set_index('file'))
print('If imu_present is far below 100%, an older batch may have been built without IMU;')
print('those rows are dropped from training and scored with head = 0 at test.')

## 3. IMU axes: calibration or fit

The gyro is stored in the IMU's own frame, so "yaw" and "pitch" mean nothing until it is
rotated into the eye's axes (CPF: +X left, +Y up, +Z forward). Two ways:

| | how | trust |
|---|---|---|
| **calib** | Aria's factory calibration, read from one VRS by `eyehead.imu_calib` | exact (devices share one design, so one VRS serves all) |
| **fit** | least squares on the training rows, assuming the eyes mostly counter-rotate the head | good if VOR dominates, but makes the VOR sign check circular |

This cell uses `imu_calib.json` on Drive if it exists. Otherwise it tries a VRS: set
`VRS_PATH`, or keep the AEA download-links JSON at `URLS_JSON` and it downloads the smallest
VRS once (usually a GB or two). With neither, training falls back to **fit**.

In [ ]:
VRS_PATH  = None                                             # e.g. '/content/raw/<seq>/<file>.vrs'
URLS_JSON = os.path.join(DRIVE_DIR, 'aea_download_urls.json')  # only used to fetch one VRS

if os.path.exists(IMU_CALIB):
    print('using', IMU_CALIB)
    print(open(IMU_CALIB).read())
else:
    if VRS_PATH is None and os.path.exists(URLS_JSON):
        import urllib.request
        meta = json.load(open(URLS_JSON))
        seq, e = min(((s, v) for s, v in meta['sequences'].items() if 'main_vrs' in v),
                     key=lambda kv: kv[1]['main_vrs']['file_size_bytes'])
        VRS_PATH = os.path.join('/content/raw', seq, e['main_vrs']['filename'])
        os.makedirs(os.path.dirname(VRS_PATH), exist_ok=True)
        if not os.path.exists(VRS_PATH):
            print(f"downloading the smallest VRS: {seq} ({e['main_vrs']['file_size_bytes']/1e9:.2f} GB)")
            urllib.request.urlretrieve(e['main_vrs']['download_url'], VRS_PATH)
    if VRS_PATH and os.path.exists(VRS_PATH):
        sh('pip -q install projectaria-tools', tail=3)
        sh(f'python -u -m eyehead.imu_calib --vrs "{VRS_PATH}" --out "{IMU_CALIB}"')
    else:
        print(f'no imu_calib.json, no VRS_PATH and no {URLS_JSON}:')
        print('   the IMU map will be FIT from each fold\'s training rows.')

IMU_ARGS = (f'--imu_rot calib --imu_calib "{IMU_CALIB}"' if os.path.exists(IMU_CALIB)
            else '--imu_rot fit')
print('\nIMU_ARGS =', IMU_ARGS)

## 4. Look at the features before training

If the four classes do not differ in these three signals, no model will separate them, and
it is better to know that now. Read three things:

1. **VOR sign check.** When the head turns and the eye moves smoothly, cos should be clearly
   negative. A positive median means a sign or axis in the IMU map is wrong (in **fit** mode
   it is negative by construction, so it proves nothing there).
2. **Head speed** should be low for STABLE and REFIXATION and high for PURSUIT and
   TRANSITION.
3. **Eye speed max** (the saccade proxy) should be high for REFIXATION and TRANSITION.

In [ ]:
from eyehead.features import (fit_head_map, compute_channels, vor_check, class_profile,
                              load_imu_calib)

f1 = load_rows(os.path.join(FOLDS, 'train_val_fold1.csv'))
E, G, M, ok = raw_signals(f1)
y = f1.quad.to_numpy()
tr = (f1.split == 'train').to_numpy() & ok & M.any(1)

if os.path.exists(IMU_CALIB):
    A, _ = load_imu_calib(IMU_CALIB)
    print('IMU map: calibration')
else:
    A, info = fit_head_map(E[tr], G[tr], M[tr], ok[tr])
    print(f"IMU map: fit on fold-1 training rows   VOR gain {np.round(info['vor_gain'], 2)}"
          f"   R2 {info['r2']:.3f}   ({info['n_steps']:,} steps)")
print('raw gyro -> [head_yaw, head_pitch]:')
print(np.round(A, 3))

X, H = compute_channels(E, G, M, A)
v = vor_check(E[tr], H[tr], M[tr], ok[tr])
print(f"\nVOR sign check: median cos {v['median_cos']:+.3f} over {v['n']:,} steps, "
      f"{100*v['frac_negative']:.0f}% negative   "
      f"[{'ok' if v['median_cos'] < -0.3 else 'CHECK THE IMU MAP'}]")

print('\nper-class profile, fold-1 training rows (raw units):')
prof = class_profile(X[tr], M[tr], y[tr])
display(prof.round(3))

# per-row summaries for plotting
Mr = M[tr]
eye_max  = np.where(Mr, X[tr, 0], 0).max(1)
head_avg = (X[tr, 1] * Mr).sum(1) / Mr.sum(1)
cls = y[tr]
colors = {0: 'tab:blue', 1: 'tab:orange', 2: 'tab:red', 3: 'tab:green'}

fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
for a_, vals, title in ((ax[0], eye_max, 'eye speed, max step per row (rad/s)'),
                        (ax[1], head_avg, 'head speed, mean per row (rad/s)')):
    a_.boxplot([vals[cls == c] + 1e-4 for c in range(4)], showfliers=False)
    a_.set_xticks(range(1, 5)); a_.set_xticklabels(list(NAMES.values()))
    a_.set_yscale('log'); a_.set_title(title)
act = (X[tr, 2] != 0) & Mr
for c in range(4):
    vals = X[tr, 2][act & (cls == c)[:, None]]
    ax[2].hist(vals, bins=40, range=(-1, 1), histtype='step', lw=1.6, density=True,
               color=colors[c], label=NAMES[c])
ax[2].set_title('cos(eye, head), active steps'); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

rng = np.random.default_rng(0)
idx = rng.choice(len(cls), size=min(4000, len(cls)), replace=False)
plt.figure(figsize=(7, 5.5))
for c in range(4):
    s = idx[cls[idx] == c]
    plt.scatter(head_avg[s] + 1e-3, eye_max[s] + 1e-3, s=5, alpha=.35, c=colors[c], label=NAMES[c])
plt.xscale('log'); plt.yscale('log')
plt.xlabel('head speed, mean (rad/s)'); plt.ylabel('eye speed, max step (rad/s)')
plt.title('each class should sit in its own region'); plt.legend(markerscale=3, fontsize=8)
plt.show()

## 5. Train the full model (eye + head + cos), 5 folds

Per fold, the IMU map (fit mode), the normalisation and the class weights come from that
fold's training rows only. Model selection is on validation **balanced accuracy**; the CV
numbers printed at the end are the mean over the **last 5 epochs**, not the best epoch.

In [ ]:
EPOCHS, PATIENCE, BATCH, LR = 150, 25, 256, 1e-3
COMMON = (f'--folds_dir "{FOLDS}" --thresholds "{THRESH}" {IMU_ARGS} '
          f'--epochs {EPOCHS} --patience {PATIENCE} --batch_size {BATCH} --lr {LR}')

t0 = time.time()
sh(f'python -u -m eyehead.train {COMMON} --out_dir "{RUN_MAIN}"')
print(f'\ntotal {(time.time()-t0)/60:.1f} min')

### Curves and the fold spread

In [ ]:
res = json.load(open(os.path.join(RUN_MAIN, 'cv_results.json')))
H_ = res['history']

fig, ax = plt.subplots(1, 4, figsize=(21, 4.2))
for k, h in H_.items():
    ep = [r['epoch'] for r in h]
    ax[0].plot(ep, [r['tr_loss'] for r in h], lw=1.2, label=k)
    ax[1].plot(ep, [r['bal_acc'] for r in h], lw=1.3, label=k)
    ax[2].plot(ep, [r['recall_2'] for r in h], lw=1.3, label=k)
    ax[3].plot(ep, [r['recall_1'] for r in h], lw=1.3, label=k)
ax[0].set_title('train loss (weighted CE)'); ax[0].legend(fontsize=7)
ax[1].axhline(0.25, ls='--', c='k', lw=1); ax[1].set_title('val balanced accuracy (chance 0.25)')
ax[2].set_title('val REFIXATION recall'); ax[3].set_title('val PURSUIT recall')
for a_ in ax: a_.set_xlabel('epoch')
plt.tight_layout(); plt.show()

tab = pd.DataFrame([f['last'] for f in res['folds']], index=[f"fold {f['fold']}" for f in res['folds']])
tab = tab.rename(columns={'recall_0': 'rec_TRANSITION', 'recall_1': 'rec_PURSUIT',
                          'recall_2': 'rec_REFIXATION', 'recall_3': 'rec_STABLE'})
show = ['bal_acc', 'macro_f1', 'auc_gate', 'rec_TRANSITION', 'rec_PURSUIT', 'rec_REFIXATION',
        'rec_STABLE', 'gate_false_skip_rate']
display(tab[show].round(3))
print('mean +/- std over folds:')
for c in show:
    print(f'   {c:<22} {tab[c].mean():.3f} +/- {tab[c].std():.3f}')
for f in res['folds']:
    v = f['vor_check']
    print(f"   fold {f['fold']}: IMU map {f['head_map_info'].get('mode')}, VOR median cos {v['median_cos']:+.3f}")

## 6. Ablations and the shuffle control

| run | flag | question |
|---|---|---|
| eye only | `--streams eye` | how far does gaze alone get in this model? |
| eye + head | `--streams eye,head` | what does the IMU add? |
| full | (section 5) | what does cos(eye, head) add on top? |
| shuffle control | `--shuffle_control` | features paired with the wrong labels: **must** fall to ≈ 0.25 balanced accuracy, or the setup leaks |

All variants train on the same rows. Only the last lines of each run are shown.

In [ ]:
VARIANTS = {
    'eye_only':  '--streams eye',
    'eye_head':  '--streams eye,head',
    'shuffle':   '--shuffle_control',
}
for name, flag in VARIANTS.items():
    out = os.path.join(RUNS, f'eyehead_{name}')
    print(f"\n{'='*30} {name} {'='*30}")
    sh(f'python -u -m eyehead.train {COMMON} {flag} --out_dir "{out}"', tail=14)

rows = {}
for name, path in [('eye_only', os.path.join(RUNS, 'eyehead_eye_only')),
                   ('eye_head', os.path.join(RUNS, 'eyehead_eye_head')),
                   ('full',     RUN_MAIN),
                   ('shuffle',  os.path.join(RUNS, 'eyehead_shuffle'))]:
    r = json.load(open(os.path.join(path, 'cv_results.json')))
    L = pd.DataFrame([f['last'] for f in r['folds']])
    rows[name] = {'bal_acc': f"{L.bal_acc.mean():.3f} +/- {L.bal_acc.std():.3f}",
                  'macro_f1': round(L.macro_f1.mean(), 3), 'auc_gate': round(L.auc_gate.mean(), 3),
                  'rec_PURSUIT': round(L.recall_1.mean(), 3), 'rec_REFIX': round(L.recall_2.mean(), 3),
                  'rec_STABLE': round(L.recall_3.mean(), 3),
                  'false_skip': round(L.gate_false_skip_rate.mean(), 3)}
cmp = pd.DataFrame(rows).T
display(cmp)

sh_bal = float(cmp.loc['shuffle', 'bal_acc'].split()[0])
print(f"shuffle control balanced accuracy {sh_bal:.3f}  "
      f"[{'PASS: near chance' if sh_bal < 0.32 else 'FAIL: the control learned something -- look for a leak'}]")

## 7. Test, once

The five fold models are averaged. Each scores the test rows with its own IMU map and
normalisation. The script prints per-model and ensemble metrics, the confusion matrix, the
skip-threshold sweep and, if the previous gate's `test_preds.csv` is on Drive, a
side-by-side on the same rows.

In [ ]:
PRED = os.path.join(RUN_MAIN, 'test_preds.csv')
base_flag = f'--baseline_preds "{BASELINE}"' if os.path.exists(BASELINE) else ''
sh(f'python -u -m eyehead.infer --folds_dir "{FOLDS}" --thresholds "{THRESH}" '
   f'--ckpt_dir "{RUN_MAIN}" --out_csv "{PRED}" {base_flag}')

### The test result in pictures

In [ ]:
pr = pd.read_csv(PRED)
summ = json.load(open(os.path.splitext(PRED)[0] + '_summary.json'))
labels = list(NAMES.values())
cm = np.array(summ['confusion'])
cm_norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)

fig, ax = plt.subplots(1, 3, figsize=(19, 5))
for a_, mat, title, fmt in ((ax[0], cm, 'counts', '{:,.0f}'),
                            (ax[1], cm_norm, 'row-normalised (diagonal = recall)', '{:.2f}')):
    a_.imshow(mat, cmap='Blues')
    a_.set_xticks(range(4)); a_.set_xticklabels(labels, rotation=30, ha='right')
    a_.set_yticks(range(4)); a_.set_yticklabels(labels)
    for i in range(4):
        for j in range(4):
            a_.text(j, i, fmt.format(mat[i, j]), ha='center', va='center', fontsize=9,
                    color='white' if mat[i, j] > mat.max() / 2 else 'black')
    a_.set_xlabel('predicted'); a_.set_ylabel('truth'); a_.set_title(title)

ens = summ['ensemble']
new_rec = [ens[f'recall_{c}'] for c in range(4)]
x = np.arange(4)
if summ.get('baseline'):
    old_rec = [summ['baseline'][f'recall_{c}'] for c in range(4)]
    ax[2].bar(x - .19, old_rec, .38, label='previous gaze-only gate', color='tab:gray')
    ax[2].bar(x + .19, new_rec, .38, label='eye + head', color='tab:blue')
else:
    ax[2].bar(x, new_rec, .5, label='eye + head', color='tab:blue')
ax[2].set_xticks(x); ax[2].set_xticklabels(labels, rotation=30, ha='right')
ax[2].set_ylim(0, 1); ax[2].set_title('test recall per class'); ax[2].legend()
plt.tight_layout(); plt.show()

sw = pd.DataFrame(summ['skip_sweep'])
plt.figure(figsize=(7, 5))
plt.plot(sw.skip_pct, sw.false_skip_pct, 'o-', label='eye + head, skip iff P(STABLE) > t')
plt.scatter([100 * (1 - ens['gate_send_rate'])], [100 * ens['gate_false_skip_rate']],
            s=90, c='tab:blue', marker='*', zorder=5, label='eye + head, argmax')
if summ.get('baseline'):
    b = summ['baseline']
    plt.scatter([100 * (1 - b['gate_send_rate'])], [100 * b['gate_false_skip_rate']],
                s=90, c='tab:gray', marker='s', zorder=5, label='previous gate')
oracle = 100 * (1 - ens['gate_oracle_send_rate'])
plt.axvline(oracle, ls=':', c='k', lw=1, label=f'oracle skip rate {oracle:.0f}%')
plt.xlabel('frames skipped (%)  -> compute saved'); plt.ylabel('false skips (%)  -> content lost')
plt.title('the gate trade-off: lower-right is better'); plt.legend(fontsize=8); plt.grid(alpha=.3)
plt.show()

print(f"balanced accuracy {ens['bal_acc']:.3f}   macro F1 {ens['macro_f1']:.3f}   AUC_gate {ens['auc_gate']:.4f}")
print(f"REFIXATION recall {ens['recall_2']:.3f}  precision {ens['precision_2']:.3f}   "
      f"PURSUIT recall {ens['recall_1']:.3f}  precision {ens['precision_1']:.3f}")
print(f"skips {100*(1-ens['gate_send_rate']):.1f}%   false skip {100*ens['gate_false_skip_rate']:.1f}%")

---

## Reading it

| | |
|---|---|
| **balanced accuracy** | chance is 0.25. Read it before plain accuracy, which TRANSITION and STABLE dominate |
| **REFIXATION / PURSUIT recall** | the reason this model exists. The previous gate: 0.00 and 0.09 on test |
| **REFIXATION precision** | speed alone cannot tell "saccade and back" (STABLE) from "saccade to somewhere new" (REFIXATION); low precision here is that limitation |
| **false skip** | content lost, unrecoverable. The previous gate: 5.7% on test |
| **eye only vs eye + head** | what the IMU is worth |
| **eye + head vs full** | what cos(eye, head) is worth |
| **shuffle control** | must sit near 0.25. If not, stop and find the leak |
| **VOR sign check** | must be clearly negative in calib mode, or the head axes are wrong |

## If REFIXATION and PURSUIT stay low

1. Check section 4 first: if the classes overlap in the raw signals, the limit is the
   input, not the model.
2. Try `--class_weight sqrt` if the rare classes are over-predicted, or keep `inv` if they
   are still ignored.
3. Compare `--imu_rot calib` with `--imu_rot fit`; a wrong head map makes cos noise.
4. The labels themselves are DINOv2 cosines cut at the median, so many rows sit right at a
   class boundary. Some confusion there is label noise, not model error.

## What is not tested here

Whether a VLM answers better on the frames this gate keeps. Every number above measures
agreement with a DINOv2 similarity proxy.